## Tokenizing the Sarcasm dataset

In [8]:
# Import libraries
import os
import zipfile

import tensorflow as tf
import json
import tensorflow_datasets as tfds
from tensorflow.keras.utils import pad_sequences

In [5]:
!kaggle datasets download rmisra/news-headlines-dataset-for-sarcasm-detection

Dataset URL: https://www.kaggle.com/datasets/rmisra/news-headlines-dataset-for-sarcasm-detection
License(s): Attribution 4.0 International (CC BY 4.0)
  0%|                                               | 0.00/3.30M [00:00<?, ?B/s]
100%|██████████████████████████████████████| 3.30M/3.30M [00:00<00:00, 2.38GB/s]


In [9]:
filepath = os.path.join(os.getcwd(), "Data/news-headlines-dataset-for-sarcasm-detection.zip")

with zipfile.ZipFile(filepath, 'r') as zipref:
    zipref.extractall("Data/")

In [22]:
# Load the data from json file
json_filepath = os.path.join(os.getcwd(), "Data/Sarcasm_Headlines_Dataset.json")
datastore = [json.loads(line) for line in open(json_filepath, 'r')]

print(f"Sample data points -\n{datastore[0]}\n\n{datastore[1]}")

Sample data points -
{'article_link': 'https://www.huffingtonpost.com/entry/versace-black-code_us_5861fbefe4b0de3a08f600d5', 'headline': "former versace store clerk sues over secret 'black code' for minority shoppers", 'is_sarcastic': 0}

{'article_link': 'https://www.huffingtonpost.com/entry/roseanne-revival-review_us_5ab3a497e4b054d118e04365', 'headline': "the 'roseanne' revival catches up to our thorny political mood, for better and worse", 'is_sarcastic': 0}


### Getting and pre-processing sentences

    1. Use the TextVectorization layer for creating the vocabulary
    2. Create post-padded sequences
    3. Create pre-padded sequences

In [25]:
# Fetch the sentences (headlines)
sentences = [datapoint['headline'] for datapoint in datastore]

sentences[1]

"the 'roseanne' revival catches up to our thorny political mood, for better and worse"

#### Create tokens and post-padded sequences

In [28]:
# Initialize the layer
vectorize_layer = tf.keras.layers.TextVectorization()
vectorize_layer.adapt(sentences)

# Post padded sequences
post_padded_sequences = vectorize_layer(sentences)
vocabulary = vectorize_layer.get_vocabulary(include_special_tokens=True)

In [32]:
# Print the sentence and post-padded sequence
ind = 2
print(f"Sentence: {sentences[ind]}")
print(f"Post-padded tokenized sequence: {post_padded_sequences[ind]}")

print(f"\nShape of padded sequences: {post_padded_sequences.shape}")

Sentence: mom starting to fear son's web series closest thing she will have to grandchild
Post-padded tokenized sequence: [  140   825     2   813  1100  2048   571  5057   199   139    39    46
     2 13050     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0]

Shape of padded sequences: (26709, 39)


#### Create ragged tokens and pre-padded sequences

For prepadding, you have to setup the `TextVectorization` layer differently. You don't want to have the automatic postpadding shown above, and instead have sequences with variable length. Then, you will pass it to the `pad_sequences()` utility function you used in the previous lab. The cells below show one way to do it:

* First, you will initialize the `TextVectorization` layer and set its `ragged` flag to `True`. This will result in a [ragged tensor](https://www.tensorflow.org/guide/ragged_tensor) which simply means a tensor with variable-length elements. The sequences will indeed have different lengths after removing the zeroes, thus you will need the ragged tensor to contain them.

* Like before, you will use the layer's `adapt()` method to generate a vocabulary.

* Then, you will apply the layer to the string sentences to generate the integer sequences. As mentioned, this will not be post-padded.

* Lastly, you will pass this ragged tensor to the [pad_sequences()](https://www.tensorflow.org/api_docs/python/tf/keras/utils/pad_sequences) function to generate pre-padded sequences.

In [33]:
# Instantiate the layer with ragged flag set to true
vectorize_layer = tf.keras.layers.TextVectorization(ragged=True)

vectorize_layer.adapt(sentences)
ragged_sequences = vectorize_layer(sentences)

In [35]:
# Print a sample headline and sequence
index = 2
print(f'Sample headline: {sentences[index]}')
print(f'Ragged sequence: {ragged_sequences[index]}\n')

# Print dimensions of padded sequences
print(f'Shape of ragged sequences: {ragged_sequences.shape}')

Sample headline: mom starting to fear son's web series closest thing she will have to grandchild
Ragged sequence: [  140   825     2   813  1100  2048   571  5057   199   139    39    46
     2 13050]

Shape of ragged sequences: (26709, None)


In [38]:
# Pre-pad the ragged sequences
pre_padded_sequences = tf.keras.utils.pad_sequences(ragged_sequences.numpy(), padding='pre')

pre_padded_sequences[2]

array([    0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,   140,   825,
           2,   813,  1100,  2048,   571,  5057,   199,   139,    39,
          46,     2, 13050], dtype=int32)

In [39]:
# Print a sample headline and sequence
index = 2
print(f'Sample headline: {sentences[index]}')
print()
print(f'Post-padded sequence: {post_padded_sequences[index]}')
print()
print(f'Pre-padded sequence: {pre_padded_sequences[index]}')
print()

# Print dimensions of padded sequences
print(f'Shape of post-padded sequences: {post_padded_sequences.shape}')
print(f'Shape of pre-padded sequences: {pre_padded_sequences.shape}')

Sample headline: mom starting to fear son's web series closest thing she will have to grandchild

Post-padded sequence: [  140   825     2   813  1100  2048   571  5057   199   139    39    46
     2 13050     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0]

Pre-padded sequence: [    0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0   140   825     2   813  1100  2048   571  5057   199   139    39
    46     2 13050]

Shape of post-padded sequences: (26709, 39)
Shape of pre-padded sequences: (26709, 39)
